In [0]:
from pyspark.sql.functions import col, sum as spark_sum, when

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS canada_labour_market") # Schema

dbutils.widgets.text("base_input_path", "s3a://your-bucket-name/curated/")  # please change to your own bucket

base_input = dbutils.widgets.get("base_input_path").rstrip("/")

# Read the data into Spark DataFrames
df_monthly  = spark.read.parquet(f"{base_input}/monthly_vacancies_employed_curated/")
df_naics    = spark.read.parquet(f"{base_input}/naics_employed_curated/")
df_quarterly = spark.read.parquet(f"{base_input}/quarterly_vacancies_employed_curated/")

In [0]:
df_monthly_clean = df_monthly \
    .withColumnRenamed("Job vacancies", "job_vacancies") \
    .withColumnRenamed("Job vacancy rate", "job_vacancy_rate") \
    .withColumnRenamed("Payroll employees", "payroll_employees") \
    .withColumnRenamed("Total Employment", "total_employment") \
    .withColumnRenamed("GEO", "geo") \
    .withColumnRenamed("Year", "year") \
    .withColumnRenamed("Date_parsed", "date_parsed")

df_naics_clean = df_naics \
    .withColumnRenamed("Total Employment", "total_employment") \
    .withColumnRenamed("GEO", "geo") \
    .withColumnRenamed("Year", "year") \
    .withColumnRenamed("NAICS", "naics") \
    .withColumnRenamed("Date_parsed", "date_parsed")

df_quarterly_clean = df_quarterly \
    .withColumnRenamed("Average offered hourly wage", "avg_offered_hourly_wage") \
    .withColumnRenamed("Job vacancies", "job_vacancies") \
    .withColumnRenamed("Job vacancy rate", "job_vacancy_rate") \
    .withColumnRenamed("Payroll employees", "payroll_employees") \
    .withColumnRenamed("Total Employment", "total_employment") \
    .withColumnRenamed("GEO", "geo") \
    .withColumnRenamed("Year", "year") \
    .withColumnRenamed("NAICS", "naics") \
    .withColumnRenamed("Date_parsed", "date_parsed")

In [0]:
assert df_monthly.count() == df_monthly_clean.count(), "Monthly row count mismatch after renaming"
assert df_naics.count() == df_naics_clean.count(), "NAICS row count mismatch after renaming"
assert df_quarterly.count() == df_quarterly_clean.count(), "Quarterly row count mismatch after renaming"
print("Row count is fine")

In [0]:
# Checking schema consistency
expected_monthly_cols = ["date_parsed", "geo", "year", "job_vacancies", "job_vacancy_rate", "payroll_employees", "total_employment"]
expected_naics_cols = ["date_parsed", "geo", "year", "naics", "total_employment"]
expected_quarterly_cols = ["date_parsed", "geo", "year", "naics", "job_vacancies", "job_vacancy_rate", "avg_offered_hourly_wage", "payroll_employees", "total_employment"]

for col_schema in expected_monthly_cols:
    assert col_schema in df_monthly_clean.columns, f"Missing column in monthly: {col_schema}"

for col_schema in expected_naics_cols:
    assert col_schema in df_naics_clean.columns, f"Missing column in naics: {col_schema}"

for col_schema in expected_quarterly_cols:
    assert col_schema in df_quarterly_clean.columns, f"Missing column in quarterly: {col_schema}"

print("Schema is fine")

In [0]:
def pre_delta_table_save_export(df, name, critical_cols):
    nulls = df.select([spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) 
                                    for c in df.columns]).collect()[0].asDict()

    for c in critical_cols:
        assert nulls[c] == 0, f"Nulls found in critical column: {c} ({nulls[c]} nulls)"
    print(f"{name}: No nulls in critical columns")

    dupes = df.groupBy(critical_cols) \
                  .count().filter("count > 1").orderBy("count", ascending=True)

    assert dupes.count() == 0, f"Duplicate records found: {dupes.show(truncate=False)}"
    print(f"{name}: No duplicates found\n")

naics_and_quarterly_critical_cols = ["year", "geo", "naics", "date_parsed"]
monthly_critical_cols = ["year", "geo", "date_parsed"]

pre_delta_table_save_export(df_naics_clean, "naics", naics_and_quarterly_critical_cols)
pre_delta_table_save_export(df_quarterly_clean, "quarterly", naics_and_quarterly_critical_cols)
pre_delta_table_save_export(df_monthly_clean, "monthly", monthly_critical_cols)

print("Ready for delta table")


In [0]:
# Tables are saved to Unity Catalog under workspace.canada_labour_market
# If your workspace uses a different catalog, update the schema name accordingly
df_monthly_clean.write.mode("overwrite").saveAsTable("workspace.canada_labour_market.monthly_vacancies_curated")
df_naics_clean.write.mode("overwrite").saveAsTable("workspace.canada_labour_market.naics_employed_curated")
df_quarterly_clean.write.mode("overwrite").saveAsTable("workspace.canada_labour_market.quarterly_vacancies_curated")